# AI Water Quality Forecasting System
### Project ID: P18 | Group ID: OJ18
**Mapped SIH Problem Statement**: SIH1694 – Real-time Ganga River Water Quality Forecasting using AI-enabled DSS, satellite data, IoT, and dynamic models.

This interactive notebook demonstrates the end-to-end data science and machine learning pipeline:
1. Data Ingestion & Station Metadata
2. Exploratory Data Analysis (EDA) & Limnological Statistics
3. Time-Series Feature Engineering (Harmonics, Lags, Rolling Means)
4. Chronological Model Training (Linear, Ridge, Random Forest, Gradient Boosting)
5. Multi-Horizon Recursive Forecasting & 95% Confidence Intervals
6. Water Quality Index (WQI) & Pollution Risk Assessment

In [ ]:
import sys
from pathlib import Path

# Set project root and paths
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from data_loader import load_raw_data, load_stations_metadata, get_dataset_summary
from preprocessing import handle_missing_and_outliers, create_time_series_features, chronological_train_test_split
from eda import compute_statistical_summary, compute_station_rankings, plot_parameter_timeseries, plot_correlation_heatmap
from train_model import train_and_compare_models
from forecasting import generate_forecast, plot_forecast_chart
from risk_detection import calculate_water_quality_index, classify_risk, detect_parameter_anomalies

print("All system modules loaded successfully!")

## 1. Data Ingestion & Station Metadata

In [ ]:
raw_data = load_raw_data()
stations_df = load_stations_metadata()
summary = get_dataset_summary(raw_data)

print(f"Total Records: {summary['total_records']:,}")
print(f"Date Range: {summary['date_range'][0]} to {summary['date_range'][1]}")
print(f"Monitoring Stations: {len(summary['stations'])}")
display(stations_df)

## 2. Statistical Analysis & Limnological Summary

In [ ]:
stats_df = compute_statistical_summary(raw_data)
display(stats_df)

rankings = compute_station_rankings(raw_data)
print("\n--- Station Average Water Quality Levels ---")
display(rankings)

## 3. Parameter Correlation Matrix (Henry's Law & Organic Coupling)

In [ ]:
fig_corr = plot_correlation_heatmap(raw_data)
fig_corr.show()

## 4. Time-Series Feature Engineering

In [ ]:
clean_df = handle_missing_and_outliers(raw_data)
feat_df = create_time_series_features(clean_df, target_param="dissolved_oxygen")
print("Engineered Features Matrix Shape:", feat_df.shape)
print("Features Sample:")
display(feat_df[["date", "station_name", "dissolved_oxygen", "sin_day", "cos_day", "dissolved_oxygen_lag_1", "dissolved_oxygen_lag_7"]].head())

## 5. Machine Learning Model Training & Comparative Evaluation

In [ ]:
best_mod, best_name, metrics, scaler, feat_cols, comp_df = train_and_compare_models(clean_df, target_param="dissolved_oxygen")
print(f"Selected Best Model: {best_name}")
display(comp_df)

## 6. Multi-Horizon Recursive Forecasting (7-Day Ahead at Kanpur)

In [ ]:
kanpur_data = clean_df[clean_df["station_name"] == "Kanpur (Jajmau)"]
fc_df = generate_forecast(kanpur_data, target_param="dissolved_oxygen", horizon_days=7)
display(fc_df)

fig_fc = plot_forecast_chart(kanpur_data, fc_df, parameter="dissolved_oxygen", station_name="Kanpur (Jajmau)")
fig_fc.show()

## 7. Water Quality Index (WQI) & Pollution Risk Classification

In [ ]:
latest_obs = kanpur_data.iloc[-1]
wqi, sub_indices = calculate_water_quality_index(latest_obs)
risk = classify_risk(wqi)
anomalies = detect_parameter_anomalies(latest_obs)

print(f"Station: Kanpur (Jajmau)")
print(f"Calculated WQI: {wqi} ({risk['badge']})")
print(f"CPCB Designated Equivalent: {risk['cpcb_class']}")
print(f"Description: {risk['description']}")
print("\nDetected Anomalies:")
for a in anomalies:
    print(f" - [{a['severity']}] {a['message']}")